# Stock Price Prediction using Random Forest Regression
This notebook provides a walkthrough of the Stock Price Prediction system, demonstrating data collection, feature engineering, model training, evaluation, and future forecasting.

In [ ]:
import os
import sys
# Add root path to Python path so we can import our modules
sys.path.append(os.path.abspath('..'))

In [ ]:
import pandas as pd
import numpy as np
from src.data_collector import StockDataCollector
from src.feature_engineer import FeatureEngineer
from src.model_pipeline import StockPredictionModel
from src.visualizer import StockVisualizer

## 1. Data Collection
We will fetch 5 years of historical stock data for Microsoft (MSFT) using yfinance.

In [ ]:
ticker = "MSFT"
collector = StockDataCollector(cache_dir="../data")
df_raw = collector.fetch_data(ticker=ticker, years=5, force_download=False)
df_raw.head()

## 2. Feature Engineering & Preprocessing
We clean the data, calculate technical indicators (SMA, EMA, RSI, MACD, Bollinger Bands, Returns, Volatility, Volume Change), and create the target variable (next day's closing price).

In [ ]:
engineer = FeatureEngineer()
df_clean = engineer.preprocess_data(df_raw)
df_features = engineer.add_technical_indicators(df_clean)
df_dataset = engineer.create_target_variable(df_features)
print(f"Dataset shape: {df_dataset.shape}")

## 3. Train-Test Split & Feature Scaling
Split the data chronologically (first 80% train, last 20% test) to prevent data leakage.

In [ ]:
X_train, y_train, X_test, y_test = engineer.split_data(df_dataset, train_ratio=0.8)
X_train_scaled, X_test_scaled = engineer.scale_features(X_train, X_test, fit=True)

## 4. Model Training & Tuning
We train the Random Forest Regressor and tune the parameters using TimeSeriesSplit cross validation.

In [ ]:
model = StockPredictionModel(random_state=42)
# Let's do a fast grid search to demonstrate hyperparameter tuning
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [10, None],
    'min_samples_split': [2, 5]
}
model.tune_hyperparameters(X_train_scaled, y_train, param_grid=param_grid)

## 5. Model Evaluation
Evaluate metrics (MAE, MSE, RMSE, R2, MAPE) on the test dataset.

In [ ]:
metrics = model.evaluate(X_test_scaled, y_test)
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

## 6. Visualization
We generate performance and diagnostic plots using `StockVisualizer`.

In [ ]:
visualizer = StockVisualizer(output_dir="../images")
predictions = model.predict(X_test_scaled)

# 1. Actual vs Predicted
fig1 = visualizer.plot_actual_vs_predicted(y_test, predictions, ticker, save=True)

# 2. Feature Importances
df_importance = model.get_feature_importances(engineer.feature_cols)
fig2 = visualizer.plot_feature_importance(df_importance, ticker, save=True)

# 3. Residual Error Distribution
fig3 = visualizer.plot_residuals(y_test, predictions, ticker, save=True)

# 4. Feature Correlation
fig4 = visualizer.plot_correlation_heatmap(df_features[engineer.feature_cols], ticker, save=True)

## 7. Model Saving & Loading

In [ ]:
model_path = f"../models/{ticker.lower()}_rf_model.pkl"
model.save_model(model_path, engineer)

# Test loading
loaded_model, loaded_engineer = StockPredictionModel.load_model(model_path)
print("Model loaded successfully!")

## 8. Recursive 7-Day Future Forecasting
We forecast the next 7 business trading days using our trained model.

In [ ]:
df_forecast = loaded_model.predict_future_days(df_features, loaded_engineer, days=7)
df_forecast